<a href="https://colab.research.google.com/github/Rushikesh042/VLM-Dental-Classification/blob/main/Stage0_Decision_Gates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 0 Decision Gates: Crestal Precision and Label-Blind Biomarker Viability

In [ ]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 70.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---- stdlib ----
import json
import os
import pathlib
import warnings

# Silence known-benign third-party deprecation chatter BEFORE importing torch/monai;
# some of these fire at import time, so they must be registered first.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*cuda.cudart.*")
warnings.filterwarnings("ignore", message=".*non-tuple sequence for multidimensional indexing.*")

# ---- third-party ----
import matplotlib.pyplot as plt
import monai.networks.nets
import monai.inferers
import numpy as np
import pandas as pd
import scipy.ndimage
import scipy.stats
import sklearn.metrics
import torch

PROJECT_ROOT = pathlib.Path(os.getenv("DENTAL_PROJECT_ROOT",
    "/content/drive/MyDrive/Teeth_Segmentation_Classification"))

PRE = PROJECT_ROOT / "Preprocessed_Dataset"
MMD_PRE = PRE / "MMDental"


MMD_TEACHER_DIR = MMD_PRE / "images_teacher_space"
MMD_MANIFEST = MMD_PRE / "manifests" / "manifest.csv"

S1_4CLASS = PROJECT_ROOT / "Stage1_ToothFairy2_SegResNet"
S1_4CLASS_CKPT = S1_4CLASS / "checkpoints" / "stage1_segresnet_best_full.pt"
S1_4CLASS_CFG = S1_4CLASS / "logs" / "stage1_model_config.json"
TF2_TEST_SPLIT = S1_4CLASS / "splits" / "stage1_tf2_test.csv"

BRIDGE = PROJECT_ROOT / "Stage1_Inference_MMDental"
TEACHER_SEG = BRIDGE / "teacher_seg"
GEOM_DIR = BRIDGE / "geometry"              # geometry sidecars live here, not under MMDental
BRIDGE_QC_CSV = (
    BRIDGE
    / "stage1_transfer_qc.csv"
)
STAGE_REST_ROOT = PROJECT_ROOT / "Stage2_to_Stage6_Biomarker_Arbiter_QA"
SUBDIRS = ("qc", "biomarkers", "arbiter", "classification", "qa",
           "figures", "tables", "logs", "reports")
for d in SUBDIRS:
    (STAGE_REST_ROOT / d).mkdir(parents=True, exist_ok=True)

def _optional_path(env_name):
    raw = os.getenv(env_name, "").strip()
    return pathlib.Path(raw) if raw else None      # unset/empty -> None, not Path('.')

def _present(p):
    return p is not None and p.exists()

CLEAN_REFERENCE_CSV = _optional_path("CLEAN_REFERENCE_CSV")
LANDMARK_REFERENCE_CSV = _optional_path("LANDMARK_REFERENCE_CSV")

TEACHER_SPACING = (0.4, 0.4, 0.4)           # ToothFairy2 / teacher space, uniform isotropic
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SUPPORTS_BF16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)
SEED = int(os.getenv("STAGE0_SEED", "42"))
np.random.seed(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("device:", DEVICE)

PROJECT_ROOT: /content/drive/MyDrive/Teeth_Segmentation_Classification
device: cuda


In [ ]:
REQUIRED = {
    "MMDental manifest": MMD_MANIFEST,
    "MMDental teacher images": MMD_TEACHER_DIR,
    "Stage-I 4-class checkpoint": S1_4CLASS_CKPT,
    "Stage-I 4-class config": S1_4CLASS_CFG,
    "TF2 held-out test split": TF2_TEST_SPLIT,
    "Bridge teacher_seg dir": TEACHER_SEG,
}
missing = {k: str(v) for k, v in REQUIRED.items() if not v.exists()}
if missing:
    raise FileNotFoundError("Missing required Stage-I/preprocessing artifacts:\n" +
                            "\n".join(f"  {k}: {p}" for k, p in missing.items()))

n_seg4 = len(list(TEACHER_SEG.glob("*_4class.npy")))
n_grp = len(list(TEACHER_SEG.glob("*_fdi.npy")))   # holds 0..4 group codes (legacy filename)
n_geom = len(list(GEOM_DIR.glob("*.json"))) if GEOM_DIR.exists() else 0
print(f"teacher_seg: {n_seg4} 4class, {n_grp} group masks | geometry sidecars: {n_geom}")
print("clean reference:", "PRESENT" if _present(CLEAN_REFERENCE_CSV) else "ABSENT (Step 0 still runs)")
print("landmark reference:", "PRESENT" if _present(LANDMARK_REFERENCE_CSV) else "ABSENT (proxy mode)")
EXPECTED_MMD_CASES = len(pd.read_csv(MMD_MANIFEST))

assert n_seg4 == EXPECTED_MMD_CASES, (
    f"Expected {EXPECTED_MMD_CASES} 4-class masks, "
    f"found {n_seg4}"
)

assert n_grp == EXPECTED_MMD_CASES, (
    f"Expected {EXPECTED_MMD_CASES} tooth-group masks, "
    f"found {n_grp}"
)

assert n_geom == EXPECTED_MMD_CASES, (
    f"Expected {EXPECTED_MMD_CASES} geometry sidecars, "
    f"found {n_geom}"
)

teacher_seg: 403 4class, 403 group masks | geometry sidecars: 403
clean reference: ABSENT (Step 0 still runs)
landmark reference: ABSENT (proxy mode)


In [ ]:
# ---- FIREWALL: biomarker inputs are masks/geometry only, never labels or metadata ----
# Any column derived from the clinical record is forbidden inside biomarker computation.
FORBIDDEN_BIOMARKER_FIELDS = {
    "binary_label", "binary_label_clean", "periodontitis_label", "label",
    "label_source", "label_confidence", "record_label_quality", "evidence_source",
    "sample_weight", "age", "sex", "diagnosis_text", "treatment_text", "qa_answer",
}

def assert_firewall(used_fields):
    leaked = set(used_fields) & FORBIDDEN_BIOMARKER_FIELDS
    if leaked:
        raise RuntimeError(f"FIREWALL VIOLATION: biomarker touched forbidden fields {sorted(leaked)}")

# The biomarker functions below take ONLY (seg4, group_mask, conf4, spacing). They have no
# access to the manifest. Label fields are read separately and joined AFTER computation,
# only for the reliable-subset comparison in 0C.
print("Firewall active. Forbidden fields:", len(FORBIDDEN_BIOMARKER_FIELDS))

Firewall active. Forbidden fields: 14


In [ ]:
# ---- load MMDental manifest; robustly detect label and confidence/source columns ----
manifest = pd.read_csv(MMD_MANIFEST)

manifest["patient_id"] = (
    manifest["patient_id"]
    .astype(str)
    .str.strip()
)

manifest_ids = set(
    manifest["patient_id"]
)

seg4_ids = {
    path.name.removesuffix("_4class.npy")
    for path in TEACHER_SEG.glob("*_4class.npy")
}

group_ids = {
    path.name.removesuffix("_fdi.npy")
    for path in TEACHER_SEG.glob("*_fdi.npy")
}

geometry_ids = {
    path.stem
    for path in GEOM_DIR.glob("*.json")
}

teacher_image_ids = {
    path.stem
    for path in MMD_TEACHER_DIR.glob("*.npy")
}

assert seg4_ids == manifest_ids, (
    "4-class mask IDs do not match manifest IDs. "
    f"Missing: {sorted(manifest_ids - seg4_ids)[:10]} | "
    f"Extra: {sorted(seg4_ids - manifest_ids)[:10]}"
)

assert group_ids == manifest_ids, (
    "Tooth-group mask IDs do not match manifest IDs. "
    f"Missing: {sorted(manifest_ids - group_ids)[:10]} | "
    f"Extra: {sorted(group_ids - manifest_ids)[:10]}"
)

assert geometry_ids == manifest_ids, (
    "Geometry IDs do not match manifest IDs. "
    f"Missing: {sorted(manifest_ids - geometry_ids)[:10]} | "
    f"Extra: {sorted(geometry_ids - manifest_ids)[:10]}"
)

assert teacher_image_ids == manifest_ids, (
    "Teacher-image IDs do not match manifest IDs. "
    f"Missing: {sorted(manifest_ids - teacher_image_ids)[:10]} | "
    f"Extra: {sorted(teacher_image_ids - manifest_ids)[:10]}"
)

print(
    "[ok] manifest, teacher images, masks, "
    "and geometry contain identical patient sets"
)

LABEL_CANDIDATES = ["binary_label_clean", "binary_label", "periodontitis_label", "label"]
SOURCE_CANDIDATES = ["label_source", "evidence_source"]
CONF_CANDIDATES = ["record_label_quality", "label_confidence"]

label_col = next((c for c in LABEL_CANDIDATES if c in manifest.columns), None)
source_col = next((c for c in SOURCE_CANDIDATES if c in manifest.columns), None)
conf_col = next((c for c in CONF_CANDIDATES if c in manifest.columns), None)
print("detected -> label:", label_col, "| source:", source_col, "| confidence:", conf_col)
if label_col is None:
    warnings.warn("No binary label column found; 0C reliable-subset comparison will be skipped.")

# Reliable subsets for the 0C viability test (used ONLY after biomarker computation).
# Reliable positive: ICD-coded periodontitis. Reliable negative: ICD gingival/other at high conf.
def _to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )


def reliable_subsets(df):
    if {"reliable_pos", "reliable_neg"}.issubset(df.columns):
        return (
            _to_bool(df["reliable_pos"]),
            _to_bool(df["reliable_neg"]),
        )

    if source_col is None:
        return (
            pd.Series(False, index=df.index),
            pd.Series(False, index=df.index),
        )

    src = df[source_col].astype(str)
    pos = src.eq("icd_perio_positive")
    neg = src.isin(
        {
            "icd_gingivitis_or_gingival",
            "icd_other_only",
        }
    )

    if conf_col is not None:
        neg = neg & df[conf_col].astype(str).eq("high")

    return pos, neg

rel_pos, rel_neg = reliable_subsets(manifest)
print(f"reliable positives: {int(rel_pos.sum())} | reliable negatives: {int(rel_neg.sum())}")

[ok] manifest, teacher images, masks, and geometry contain identical patient sets
detected -> label: binary_label_clean | source: label_source | confidence: label_confidence
reliable positives: 23 | reliable negatives: 291


In [ ]:
def load_spacing(patient_id):
    """Load mandatory teacher-space spacing from the bridge sidecar."""
    sidecar = GEOM_DIR / f"{patient_id}.json"

    if not sidecar.exists():
        raise FileNotFoundError(
            f"Missing geometry sidecar for patient {patient_id}: "
            f"{sidecar}"
        )

    with open(sidecar, "r", encoding="utf-8") as file:
        geometry = json.load(file)

    if "spacing" not in geometry:
        raise KeyError(
            f"Geometry sidecar lacks 'spacing' for patient "
            f"{patient_id}: {sidecar}"
        )

    spacing = tuple(
        float(value)
        for value in geometry["spacing"]
    )

    if len(spacing) != 3 or not all(
        value > 0
        for value in spacing
    ):
        raise ValueError(
            f"Invalid spacing for patient {patient_id}: "
            f"{spacing}"
        )

    if any(
        abs(found - expected) > 1e-3
        for found, expected in zip(
            spacing,
            TEACHER_SPACING,
        )
    ):
        raise ValueError(
            f"Unexpected spacing for patient {patient_id}: "
            f"{spacing}; expected {TEACHER_SPACING}"
        )

    return spacing

def surface_voxels(mask):
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return np.zeros_like(mask)
    return mask & ~scipy.ndimage.binary_erosion(mask, iterations=1)

def crestal_surface_distances(pred_jaw, gt_jaw, teeth_gt, spacing, band_mm=5.0):
    """Symmetric surface distance (mm) between predicted and GT jaw surfaces,
    restricted to the crestal band within band_mm of the teeth. Returns None if undefined."""
    if not (teeth_gt.any() and gt_jaw.any() and pred_jaw.any()):
        return None
    dist_to_teeth = scipy.ndimage.distance_transform_edt(~teeth_gt, sampling=spacing)
    band = dist_to_teeth <= band_mm
    gt_s = surface_voxels(gt_jaw) & band
    pr_s = surface_voxels(pred_jaw) & band
    if not (gt_s.any() and pr_s.any()):
        return None
    d_pred = scipy.ndimage.distance_transform_edt(~gt_s, sampling=spacing)[pr_s]
    d_gt = scipy.ndimage.distance_transform_edt(~pr_s, sampling=spacing)[gt_s]
    alld = np.concatenate([d_pred, d_gt])
    return {"assd_mm": float(alld.mean()), "hd95_mm": float(np.percentile(alld, 95)),
            "median_sd_mm": float(np.median(alld)), "max_sd_mm": float(alld.max()),
            "n_surf_pred": int(pr_s.sum()), "n_surf_gt": int(gt_s.sum())}

def bone_support_proxy(seg4, group_mask, conf4, spacing, contact_mm=2.0, thr=0.5):
    """Label-blind bone-support proxy. Higher = bone hugs teeth (healthy);
    lower = recession (periodontitis-like). Inputs are masks/geometry only."""
    teeth = seg4 == 1
    jaws = seg4 == 2
    if not teeth.any() or not jaws.any():
        return None
    dist_to_jaw = scipy.ndimage.distance_transform_edt(~jaws, sampling=spacing)
    adj = dist_to_jaw <= contact_mm
    supports = []
    for g in [x for x in np.unique(group_mask) if x != 0]:
        gm = (group_mask == g) & teeth
        if gm.sum() < 20:
            continue
        supports.append(float((adj & gm).sum()) / float(gm.sum()))
    if not supports:
        return None
    supports = np.array(supports)
    teeth_surf = surface_voxels(teeth)
    boundary_contact = float((adj & teeth_surf).sum()) / max(int(teeth_surf.sum()), 1)
    near_boundary = adj & teeth_surf
    mean_conf_boundary = (float(conf4[near_boundary].mean())
                          if (conf4 is not None and near_boundary.any()) else np.nan)
    return {"mean_support": float(supports.mean()), "min_support": float(supports.min()),
            "lq_support": float(np.percentile(supports, 25)),
            "frac_groups_below_thr": float((supports < thr).mean()),
            "boundary_contact_fraction": boundary_contact,
            "mean_jaw_conf_boundary": mean_conf_boundary, "n_groups": int(len(supports))}

def cliffs_delta(a, b):
    a = np.asarray(a)[:, None]; b = np.asarray(b)[None, :]
    return float(((a > b).sum() - (a < b).sum()) / (a.shape[0] * b.shape[1]))

print("helpers ready")

helpers ready


## Step 0A: MMDental teacher-mask visual QC

In [ ]:
def teacher_fractions(pid):
    seg4 = np.load(TEACHER_SEG / f"{pid}_4class.npy")
    n = seg4.size
    return float((seg4 == 1).sum()) / n, float((seg4 == 2).sum()) / n

available = [p.name[:-len("_4class.npy")] for p in TEACHER_SEG.glob("*_4class.npy")]
frac_rows = [{"patient_id": pid, "teeth_fraction": tf, "jaw_fraction": jf}
             for pid in available for tf, jf in [teacher_fractions(pid)]]
frac_df = pd.DataFrame(frac_rows)

sel = set()
sel |= set(frac_df.nsmallest(3, "teeth_fraction")["patient_id"])
sel |= set(frac_df.nsmallest(3, "jaw_fraction")["patient_id"])
if label_col is not None:
    pos_ids = manifest.loc[rel_pos, "patient_id"].tolist()
    neg_ids = manifest.loc[rel_neg, "patient_id"].tolist()
    sel |= set([p for p in pos_ids if p in available][:3])
    sel |= set([p for p in neg_ids if p in available][:3])
if BRIDGE_QC_CSV.exists():
    bridge_qc = pd.read_csv(BRIDGE_QC_CSV)

    bridge_qc["patient_id"] = (
        bridge_qc["patient_id"]
        .astype(str)
        .str.strip()
    )

    possible_flag_columns = [
        "flag_low_teeth",
        "low_teeth_flag",
        "flagged_low_teeth",
    ]

    flag_column = next(
        (
            column
            for column in possible_flag_columns
            if column in bridge_qc.columns
        ),
        None,
    )

    if flag_column is not None:
        low_teeth_mask = _to_bool(
            bridge_qc[flag_column]
        )

        low_teeth_ids = set(
            bridge_qc.loc[
                low_teeth_mask,
                "patient_id",
            ]
        )

        sel |= low_teeth_ids

        print(
            f"Added {len(low_teeth_ids)} "
            "bridge-flagged low-teeth cases to QC"
        )
    else:
        warnings.warn(
            "Bridge QC file exists but no recognised "
            "low-teeth flag column was found."
        )
sel = sorted(sel)

sel_df = frac_df[frac_df["patient_id"].isin(sel)].copy()
sel_df.to_csv(STAGE_REST_ROOT / "qc" / "mmdental_overlay_case_selection.csv", index=False)

def contour(ax, mask2d, color):
    if mask2d.any():
        ax.contour(mask2d, levels=[0.5], colors=[color], linewidths=0.6)

for pid in sel:
    img = np.load(MMD_TEACHER_DIR / f"{pid}.npy").astype(np.float32)
    seg4 = np.load(TEACHER_SEG / f"{pid}_4class.npy")
    grp = np.load(TEACHER_SEG / f"{pid}_fdi.npy")
    teeth, jaws = seg4 == 1, seg4 == 2
    combined_mask = teeth | jaws | (grp > 0)

    if combined_mask.any():
        axial_scores = combined_mask.sum(axis=(1, 2))
        coronal_scores = combined_mask.sum(axis=(0, 2))
        sagittal_scores = combined_mask.sum(axis=(0, 1))

        cz = int(np.argmax(axial_scores))
        cy = int(np.argmax(coronal_scores))
        cx = int(np.argmax(sagittal_scores))
    else:
        cz, cy, cx = (
            np.array(img.shape) // 2
        ).tolist()
    planes = [("axial", img[cz], teeth[cz], jaws[cz], grp[cz]),
              ("coronal", img[:, cy], teeth[:, cy], jaws[:, cy], grp[:, cy]),
              ("sagittal", img[:, :, cx], teeth[:, :, cx], jaws[:, :, cx], grp[:, :, cx])]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, im, tm, jm, gm) in zip(axes, planes):
        ax.imshow(im, cmap="gray"); contour(ax, tm, "cyan"); contour(ax, jm, "yellow")
        contour(ax, gm > 0, "red"); ax.set_title(f"{pid} {name}"); ax.axis("off")
    plt.tight_layout()
    plt.savefig(STAGE_REST_ROOT / "figures" / f"qc_overlay_{pid}.png", dpi=110, bbox_inches="tight")
    plt.close(fig)

report = ["# Step 0A: MMDental teacher-mask overlay QC", "",
          f"Cases reviewed: {len(sel)}", "",
          "Inspect each overlay: teeth (cyan) and jaw (yellow) contours should track real",
          "anatomy. If the jaw contour stops short of visible bone, the teacher is",
          "under-segmenting on MMDental and every downstream bone measurement is suspect.",
          "If the masks track bone and the low fractions reflect a tighter field of view,",
          "the transfer is acceptable.", "", "## Selected cases", sel_df.to_string(index=False)]
(STAGE_REST_ROOT / "reports" / "step0a_mask_overlay_qc.md").write_text("\n".join(report))
print(f"0A done: {len(sel)} overlays saved. Review them before trusting 0B/0C.")

Added 21 bridge-flagged low-teeth cases to QC
0A done: 30 overlays saved. Review them before trusting 0B/0C.


## Step 0B: ToothFairy2 crestal-band precision (mm)


In [ ]:
def build_4class_model():
    mc = json.load(open(S1_4CLASS_CFG))
    model = monai.networks.nets.SegResNet(
        spatial_dims=mc.get("spatial_dims", 3), init_filters=mc.get("init_filters", 16),
        in_channels=mc.get("in_channels", 1), out_channels=mc["out_channels"],
        blocks_down=tuple(mc.get("blocks_down", (1, 2, 2, 4))),
        blocks_up=tuple(mc.get("blocks_up", (1, 1, 1))),
        dropout_prob=mc.get("dropout_prob", 0.2), norm=mc.get("norm", "instance"),
        upsample_mode=mc.get("upsample_mode", "nontrainable"))
    ck = torch.load(S1_4CLASS_CKPT, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model_state_dict"] if "model_state_dict" in ck else ck)
    return model.to(DEVICE).eval(), mc

model4, mc4 = build_4class_model()
ROI = tuple(mc4.get("patch_size", (96, 96, 96)))

@torch.no_grad()
def predict_4class(img):
    x = torch.as_tensor(
        img[None, None],
        dtype=torch.float32,
        device=DEVICE,
    )

    if DEVICE.type == "cuda":
        autocast_dtype = (
            torch.bfloat16
            if SUPPORTS_BF16
            else torch.float16
        )

        with torch.autocast(
            device_type="cuda",
            dtype=autocast_dtype,
        ):
            logits = monai.inferers.sliding_window_inference(
                x,
                ROI,
                4,
                model4,
                overlap=0.25,
                mode="gaussian",
            )
    else:
        logits = monai.inferers.sliding_window_inference(
            x,
            ROI,
            4,
            model4,
            overlap=0.25,
            mode="gaussian",
        )

    return (
        logits.argmax(dim=1)[0]
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

def dice(a, b):
    a = a.astype(bool); b = b.astype(bool); s = a.sum() + b.sum()
    return float(2 * (a & b).sum() / s) if s else float("nan")

test_df = pd.read_csv(TF2_TEST_SPLIT)
required_test_columns = {
    "case_id",
    "image_path",
    "label_path",
}

missing_test_columns = (
    required_test_columns
    - set(test_df.columns)
)

if missing_test_columns:
    raise KeyError(
        "TF2 test split is missing columns: "
        f"{sorted(missing_test_columns)}"
    )

missing_test_images = test_df.loc[
    ~test_df["image_path"].map(
        lambda value: pathlib.Path(value).exists()
    ),
    ["case_id", "image_path"],
]

missing_test_labels = test_df.loc[
    ~test_df["label_path"].map(
        lambda value: pathlib.Path(value).exists()
    ),
    ["case_id", "label_path"],
]

if len(missing_test_images):
    display(missing_test_images)
    raise FileNotFoundError(
        f"{len(missing_test_images)} TF2 test images are missing"
    )

if len(missing_test_labels):
    display(missing_test_labels)
    raise FileNotFoundError(
        f"{len(missing_test_labels)} TF2 test labels are missing"
    )

print(
    f"[ok] TF2 held-out split: {len(test_df)} cases, "
    "all paths present"
)
rows = []
for _, r in test_df.iterrows():
    img = np.load(r["image_path"]).astype(np.float32)
    gt = np.load(r["label_path"]).astype(np.uint8)
    pred = predict_4class(img)
    m = crestal_surface_distances(pred == 2, gt == 2, gt == 1, TEACHER_SPACING, band_mm=5.0)
    if m is None:
        continue
    m.update({"case_id": r["case_id"], "jaw_dice": dice(pred == 2, gt == 2),
              "teeth_dice": dice(pred == 1, gt == 1)})
    rows.append(m)

per_case = pd.DataFrame(rows)
if per_case.empty:
    raise RuntimeError(
        "Step 0B produced no valid crestal-surface measurements. "
        "Check held-out labels, teacher predictions, and spacing."
    )
per_case.to_csv(STAGE_REST_ROOT / "tables" / "tf2_crestal_surface_metrics_per_case.csv", index=False)

summary = {"n_cases": int(len(per_case)),
           "median_assd_mm": float(per_case["assd_mm"].median()),
           "mean_assd_mm": float(per_case["assd_mm"].mean()),
           "median_hd95_mm": float(per_case["hd95_mm"].median()),
           "frac_hd95_le_2mm": float((per_case["hd95_mm"] <= 2.0).mean()),
           "median_jaw_dice": float(per_case["jaw_dice"].median())}
med_assd, frac_hd95 = summary["median_assd_mm"], summary["frac_hd95_le_2mm"]
if med_assd < 1.0 and frac_hd95 >= 0.8:
    gate = "PASS"
elif med_assd <= 1.3 or frac_hd95 >= 0.6:
    gate = "CAUTION"
else:
    gate = "FAIL"
summary["decision_gate"] = gate
summary_csv = STAGE_REST_ROOT / "tables" / "tf2_crestal_surface_metrics_summary.csv"
with open(summary_csv.with_suffix(".json"), "w") as f:
    json.dump(summary, f, indent=2)
pd.DataFrame([summary]).to_csv(summary_csv, index=False)

for col, fname in [("assd_mm", "tf2_crestal_assd_distribution.png"),
                   ("hd95_mm", "tf2_crestal_hd95_distribution.png")]:
    plt.figure(figsize=(7, 4)); plt.hist(per_case[col].dropna(), bins=20)
    plt.axvline(1.0 if col == "assd_mm" else 2.0, color="r", ls="--")
    plt.xlabel(col + " (mm)"); plt.ylabel("cases"); plt.title(f"TF2 crestal {col}")
    plt.tight_layout(); plt.savefig(STAGE_REST_ROOT / "figures" / fname, dpi=120); plt.close()

(STAGE_REST_ROOT / "reports" / "step0b_crestal_precision_report.md").write_text(
    f"# Step 0B: TF2 crestal precision\n\nDecision gate: **{gate}**\n\n"
    f"Median crestal ASSD {med_assd:.3f} mm, median HD95 {summary['median_hd95_mm']:.3f} mm, "
    f"{frac_hd95*100:.0f}% of cases at HD95 <= 2 mm, over {summary['n_cases']} TF2 test cases.\n\n"
    f"Gate rule: PASS if median ASSD < 1.0 mm and >=80% of HD95 <= 2 mm. This is the noise "
    f"floor of the bone-loss biomarker; if the crest cannot be placed to ~1 mm, a 1-2 mm "
    f"disease signal cannot be measured reliably.\n")
print(f"0B done. crestal gate: {gate} | median ASSD {med_assd:.3f} mm | HD95<=2mm {frac_hd95*100:.0f}%")

[ok] TF2 held-out split: 48 cases, all paths present
0B done. crestal gate: PASS | median ASSD 0.185 mm | HD95<=2mm 91%


## Step 0C: label-blind biomarker viability on MMDental

In [ ]:
bio_rows = []

for pid in available:
    seg4 = np.load(
        TEACHER_SEG / f"{pid}_4class.npy"
    )

    grp = np.load(
        TEACHER_SEG / f"{pid}_fdi.npy"
    )

    cpath = (
        TEACHER_SEG
        / f"{pid}_4class_conf.npy"
    )

    conf4 = (
        np.load(cpath).astype(np.float32)
        if cpath.exists()
        else None
    )

    teeth_present = bool((seg4 == 1).any())
    jaws_present = bool((seg4 == 2).any())

    groups_present = [
        int(value)
        for value in np.unique(grp)
        if int(value) != 0
    ]

    measurement = bone_support_proxy(
        seg4,
        grp,
        conf4,
        load_spacing(pid),
    )

    if measurement is None:
        if not teeth_present:
            reason = "no_teeth_mask"
        elif not jaws_present:
            reason = "no_jaw_mask"
        elif not groups_present:
            reason = "no_tooth_groups"
        else:
            reason = "insufficient_group_support"

        bio_rows.append({
            "patient_id": str(pid),
            "biomarker_evaluable": False,
            "biomarker_failure_reason": reason,
            "teeth_present": teeth_present,
            "jaws_present": jaws_present,
            "n_group_labels_present": len(groups_present),
        })

        continue

    measurement.update({
        "patient_id": str(pid),
        "biomarker_evaluable": True,
        "biomarker_failure_reason": "",
        "teeth_present": teeth_present,
        "jaws_present": jaws_present,
        "n_group_labels_present": len(groups_present),
    })

    bio_rows.append(measurement)


bio = pd.DataFrame(bio_rows)

assert len(bio) == len(available), (
    "Expected one biomarker-status row per bridge case: "
    f"{len(available)}, got {len(bio)}"
)

assert bio["patient_id"].is_unique, (
    "Duplicate patients in Stage-0 biomarker table"
)

assert_firewall(bio.columns)

bio.to_csv(
    STAGE_REST_ROOT
    / "biomarkers"
    / "mmdental_crude_proxy_per_patient.csv",
    index=False,
)

non_evaluable_bio = bio.loc[
    ~bio["biomarker_evaluable"]
].copy()

non_evaluable_path = (
    STAGE_REST_ROOT
    / "qc"
    / "stage0_non_evaluable_biomarker_cases.csv"
)

non_evaluable_bio.to_csv(
    non_evaluable_path,
    index=False,
)

print(
    f"Non-evaluable biomarker cases: "
    f"{len(non_evaluable_bio)} -> "
    f"{non_evaluable_path}"
)

print(
    f"biomarker status rows: {len(bio)} | "
    f"evaluable: {int(bio['biomarker_evaluable'].sum())} | "
    f"non-evaluable: "
    f"{int((~bio['biomarker_evaluable']).sum())}"
)

display(
    bio.loc[
        ~bio["biomarker_evaluable"],
        [
            "patient_id",
            "biomarker_failure_reason",
            "teeth_present",
            "jaws_present",
            "n_group_labels_present",
        ],
    ]
)


bio_eval = bio[
    bio["biomarker_evaluable"] == True
].copy()

if (
    label_col is None
    or rel_pos.sum() < 5
    or rel_neg.sum() < 5
):
    msg = (
        "Step 0C SKIPPED: fewer than 5 reliable cases "
        "per group, or no label column."
    )

    (
        STAGE_REST_ROOT
        / "reports"
        / "step0c_biomarker_viability_report.md"
    ).write_text(
        "# Step 0C\n\n" + msg
    )

    print(msg)

else:
    lab_columns = [
        "patient_id",
        label_col,
    ]

    if source_col:
        lab_columns.append(source_col)

    lab = manifest[lab_columns].copy()

    lab["patient_id"] = (
        lab["patient_id"]
        .astype(str)
        .str.strip()
    )

    lab["reliable_pos"] = rel_pos.values
    lab["reliable_neg"] = rel_neg.values

    j = bio_eval.merge(
        lab,
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )

    pos_grp = j[
        j["reliable_pos"]
    ].copy()

    neg_grp = j[
        j["reliable_neg"]
    ].copy()

    print(
        f"Evaluable reliable positives: {len(pos_grp)} | "
        f"evaluable reliable negatives: {len(neg_grp)}"
    )

    stat_rows = []

    features = [
        "mean_support",
        "min_support",
        "lq_support",
        "boundary_contact_fraction",
        "frac_groups_below_thr",
    ]

    for feat in features:
        a = neg_grp[feat].dropna().to_numpy()
        b = pos_grp[feat].dropna().to_numpy()

        if len(a) < 3 or len(b) < 3:
            continue

        if feat == "frac_groups_below_thr":
            score_pos = b
            score_neg = a
        else:
            score_pos = -b
            score_neg = -a

        y = np.concatenate([
            np.ones(len(b)),
            np.zeros(len(a)),
        ])

        scores = np.concatenate([
            score_pos,
            score_neg,
        ])

        auroc = sklearn.metrics.roc_auc_score(
            y,
            scores,
        )

        boot = []
        rng = np.random.default_rng(SEED)

        for _ in range(2000):
            index = rng.integers(
                0,
                len(y),
                len(y),
            )

            if len(np.unique(y[index])) < 2:
                continue

            boot.append(
                sklearn.metrics.roc_auc_score(
                    y[index],
                    scores[index],
                )
            )

        if boot:
            lo, hi = np.percentile(
                boot,
                [2.5, 97.5],
            )
        else:
            lo, hi = np.nan, np.nan

        mwu = scipy.stats.mannwhitneyu(
            b,
            a,
            alternative="two-sided",
        )

        stat_rows.append({
            "feature": feat,
            "n_pos": len(b),
            "n_neg": len(a),
            "median_pos": float(np.median(b)),
            "median_neg": float(np.median(a)),
            "auroc": float(auroc),
            "auroc_lo": float(lo),
            "auroc_hi": float(hi),
            "mwu_p": float(mwu.pvalue),
            "cliffs_delta": cliffs_delta(b, a),
        })

    stats = pd.DataFrame(stat_rows)

    stats_path = (
        STAGE_REST_ROOT
        / "tables"
        / "mmdental_crude_proxy_reliable_subset_stats.csv"
    )

    stats.to_csv(
        stats_path,
        index=False,
    )

    if stats.empty:
        gate0c = "FAIL"

        report_text = (
            "# Step 0C: label-blind biomarker viability\n\n"
            "Decision gate: **FAIL**\n\n"
            "No proxy feature had enough valid reliable-positive "
            "and reliable-negative observations for evaluation.\n"
        )

        (
            STAGE_REST_ROOT
            / "reports"
            / "step0c_biomarker_viability_report.md"
        ).write_text(report_text)

        print(
            "0C failed: no proxy feature had enough valid observations."
        )

    else:
        stats = (
            stats.sort_values(
                "auroc",
                ascending=False,
            )
            .reset_index(drop=True)
        )

        stats.to_csv(
            stats_path,
            index=False,
        )

        best = stats.iloc[0]

        if (
            best["auroc"] >= 0.70
            and best["auroc_lo"] > 0.5
        ):
            gate0c = "PASS"
        elif best["auroc"] >= 0.60:
            gate0c = "WEAK PASS"
        else:
            gate0c = "FAIL"

        plt.figure(figsize=(7, 4))
        plt.boxplot([
            neg_grp["mean_support"].dropna(),
            pos_grp["mean_support"].dropna(),
        ])
        plt.xticks(
            [1, 2],
            [
                "reliable neg",
                "reliable pos",
            ],
        )
        plt.ylabel("mean bone support")
        plt.title(
            "0C support proxy by reliable label"
        )
        plt.tight_layout()
        plt.savefig(
            STAGE_REST_ROOT
            / "figures"
            / "mmdental_crude_proxy_distribution.png",
            dpi=120,
        )
        plt.close()

        best_feature = best["feature"]

        neg_values = (
            neg_grp[best_feature]
            .dropna()
            .to_numpy()
        )

        pos_values = (
            pos_grp[best_feature]
            .dropna()
            .to_numpy()
        )

        if best_feature == "frac_groups_below_thr":
            score_pos = pos_values
            score_neg = neg_values
        else:
            score_pos = -pos_values
            score_neg = -neg_values

        y = np.concatenate([
            np.ones(len(pos_values)),
            np.zeros(len(neg_values)),
        ])

        scores = np.concatenate([
            score_pos,
            score_neg,
        ])

        fpr, tpr, _ = sklearn.metrics.roc_curve(
            y,
            scores,
        )

        plt.figure(figsize=(5, 5))
        plt.plot(
            fpr,
            tpr,
            label=(
                f"{best_feature} "
                f"AUROC {best['auroc']:.2f}"
            ),
        )
        plt.plot(
            [0, 1],
            [0, 1],
            "k--",
        )
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.legend()
        plt.title("0C proxy-only ROC")
        plt.tight_layout()
        plt.savefig(
            STAGE_REST_ROOT
            / "figures"
            / "mmdental_crude_proxy_roc.png",
            dpi=120,
        )
        plt.close()

        report_path = (
            STAGE_REST_ROOT
            / "reports"
            / "step0c_biomarker_viability_report.md"
        )

        report_path.write_text(
            "# Step 0C: label-blind biomarker viability\n\n"
            f"Decision gate: **{gate0c}**\n\n"
            f"Best feature: {best_feature}, "
            f"AUROC {best['auroc']:.3f} "
            f"(95% CI {best['auroc_lo']:.3f}-"
            f"{best['auroc_hi']:.3f}), "
            f"Cliff's delta "
            f"{best['cliffs_delta']:.3f}, "
            f"MWU p {best['mwu_p']:.4f}, "
            f"on {int(best['n_pos'])} reliable positives "
            f"vs {int(best['n_neg'])} reliable negatives.\n\n"
            "PASS means a label-blind image proxy already "
            "separates cases whose record labels are relatively "
            "high-confidence. It does not establish independent "
            "clinical ground truth.\n\n"
            + stats.to_string(index=False)
        )

        print(
            f"0C done. viability gate: {gate0c} | "
            f"best {best_feature} "
            f"AUROC {best['auroc']:.3f} "
            f"[{best['auroc_lo']:.3f}, "
            f"{best['auroc_hi']:.3f}]"
        )

Non-evaluable biomarker cases: 1 -> /content/drive/MyDrive/Teeth_Segmentation_Classification/Stage2_to_Stage6_Biomarker_Arbiter_QA/qc/stage0_non_evaluable_biomarker_cases.csv
biomarker status rows: 403 | evaluable: 402 | non-evaluable: 1


,patient_id,biomarker_failure_reason,teeth_present,jaws_present,n_group_labels_present
325,585,insufficient_group_support,True,True,4


Evaluable reliable positives: 23 | evaluable reliable negatives: 290
0C done. viability gate: PASS | best boundary_contact_fraction AUROC 0.771 [0.661, 0.870]


## Step 0 status summary

In [ ]:
def read_gate(path, key="Decision gate:"):
    path = pathlib.Path(path)

    if not path.exists():
        return "NOT RUN"

    try:
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines():
            if key not in line:
                continue

            if "**" in line:
                parts = line.split("**")
                if len(parts) >= 3:
                    return parts[1].strip()

            return line.split(
                key,
                maxsplit=1,
            )[1].strip()

    except Exception as exc:
        warnings.warn(
            f"Could not read gate from {path}: {exc}"
        )
        return "NOT RUN"

    return "NOT RUN"


n_biomarker_rows = int(len(bio))

n_biomarker_evaluable = int(
    bio["biomarker_evaluable"].sum()
)

n_biomarker_non_evaluable = int(
    (~bio["biomarker_evaluable"]).sum()
)

assert (
    n_biomarker_rows
    == n_biomarker_evaluable
    + n_biomarker_non_evaluable
), "Biomarker evaluability counts do not reconcile"


OVERLAY_QC_APPROVED = (
    os.getenv(
        "STAGE0_OVERLAY_QC_APPROVED",
        "0",
    ) == "1"
)


step0a_gate = (
    "PASS"
    if OVERLAY_QC_APPROVED
    else "PENDING MANUAL REVIEW"
)

step0b_gate = read_gate(
    STAGE_REST_ROOT
    / "reports"
    / "step0b_crestal_precision_report.md"
)

step0c_gate = read_gate(
    STAGE_REST_ROOT
    / "reports"
    / "step0c_biomarker_viability_report.md"
)


if (
    step0a_gate == "PASS"
    and step0b_gate == "PASS"
    and step0c_gate == "PASS"
):
    overall_gate = "PASS"

elif (
    step0a_gate == "PASS"
    and step0b_gate in {"PASS", "CAUTION"}
    and step0c_gate in {"PASS", "WEAK PASS"}
):
    overall_gate = "CAUTION"

else:
    overall_gate = "STOP"


status = {
    "step0a_overlay_qc": step0a_gate,
    "step0b_crestal_precision": step0b_gate,
    "step0c_biomarker_viability": step0c_gate,
    "overall_stage0_gate": overall_gate,

    "clean_reference": (
        "PRESENT"
        if _present(CLEAN_REFERENCE_CSV)
        else "ABSENT"
    ),

    "landmark_reference": (
        "PRESENT"
        if _present(LANDMARK_REFERENCE_CSV)
        else "ABSENT - proxy mode"
    ),

    "biomarker_status_rows": n_biomarker_rows,
    "biomarker_evaluable": n_biomarker_evaluable,
    "biomarker_non_evaluable": n_biomarker_non_evaluable,
}


status_path = (
    STAGE_REST_ROOT
    / "reports"
    / "step0_status_summary.json"
)

with open(
    status_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        status,
        file,
        indent=2,
    )


print(json.dumps(status, indent=2))
print(f"\nSaved Stage-0 status: {status_path}")


if overall_gate == "PASS":
    print(
        "\nStage 0 PASS: proceed to Stage 2."
    )

elif overall_gate == "CAUTION":
    print(
        "\nStage 0 CAUTION: proceed to Stage 2 only "
        "after documenting the limitations."
    )

else:
    print(
        "\nStage 0 STOP: do not proceed to Stage 2 "
        "until the failed or pending gate is resolved."
    )

{
  "step0a_overlay_qc": "PENDING MANUAL REVIEW",
  "step0b_crestal_precision": "PASS",
  "step0c_biomarker_viability": "PASS",
  "overall_stage0_gate": "STOP",
  "clean_reference": "ABSENT",
  "landmark_reference": "ABSENT - proxy mode",
  "biomarker_status_rows": 403,
  "biomarker_evaluable": 402,
  "biomarker_non_evaluable": 1
}

Saved Stage-0 status: /content/drive/MyDrive/Teeth_Segmentation_Classification/Stage2_to_Stage6_Biomarker_Arbiter_QA/reports/step0_status_summary.json

Stage 0 STOP: do not proceed to Stage 2 until the failed or pending gate is resolved.
